In [ ]:
import sys, os
current_dir = os.getcwd()
project_root = current_dir[:current_dir.find("src") - 1]
sys.path.insert(0, project_root)
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
csv_semi_processed_path = os.path.join(project_root, "data", "raw", "PlantsTemperature_View_original.csv")
df_c = pd.read_csv(csv_semi_processed_path, encoding='utf-8')

In [ ]:
import  jdatetime
def jalali_to_gregorian_fast(date_str):
    jy, jm, jd = map(int, date_str.split('/'))
    jdate = jdatetime.date(jy, jm, jd)
    gdate = jdate.togregorian()
    return f"{gdate.year}-{gdate.month:02d}-{gdate.day:02d}"

In [ ]:
df = df_c[(df_c['PowerPlantCode']==104)]
df = df.sort_values(by=['Date','HourNo'])
df["HourNo"] = df["HourNo"].astype(int)
df["Date"] = df["Date"].apply(jalali_to_gregorian_fast)
df["datetime"] = pd.to_datetime(df["Date"]) + pd.to_timedelta(df["HourNo"],unit='h')
df.head()

In [ ]:
is_env = df["Code"] == "SCADAF"

In [ ]:
df_sen = df[~is_env]
df_env = df[is_env]
len(df_sen),len(df_env)

In [ ]:
import numpy as np
a, b = np.unique(df['datetime'].to_numpy(), return_counts=True)
a1, b1 = np.unique(df_env['datetime'].to_numpy(), return_counts=True)
a2, b2 = np.unique(df_sen['datetime'].to_numpy(), return_counts=True)

In [ ]:
def show_date(dates):
    dates = dates.copy()
    full_range = pd.date_range(start=dates.min(), end=dates.max(), freq='h')
    df = pd.DataFrame({'datetime': full_range})
    df['Value'] = 0
    df.loc[df['datetime'].isin(dates), 'Value'] = 1
    plt.figure(figsize=(12, 6))
    plt.plot(df['datetime'], df['Value'], drawstyle='steps-post', marker='.')
    plt.xlabel('Date and Time')
    plt.ylabel('Value')
    plt.title('Date and Time Presence')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()
    return df[df['Value'] == 0]["datetime"]

In [ ]:
datestimes = show_date(a)
datess = datestimes.dt.date
print(len(datestimes),len(datess.unique()))
for d in datess.unique():
    t = datestimes[datess == d].dt.hour.to_list()
    print(d,len(t),t)

In [ ]:
datestimes = show_date(a1)
datess = datestimes.dt.date
print(len(datestimes),len(datess.unique()))
for d in datess.unique():
    t = datestimes[datess == d].dt.hour.to_list()
    print(d,len(t),t)

In [ ]:
datestimes = show_date(a2)
datess = datestimes.dt.date
print(len(datestimes),len(datess.unique()))
for d in datess.unique():
    t = datestimes[datess == d].dt.hour.to_list()
    print(d,len(t),t)

In [ ]:
len(df_sen),len(df_env)

In [ ]:
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df_s = pd.read_csv(csv_semi_processed_path, encoding='utf-8')

In [ ]:
#t = "1400/01/01"
#df_sen = df_sen[df_sen["Date"] == t]
#df_env = df_env[df_env["Date"] == t]

In [ ]:
tem_sen = df_sen["Value"].values
tem_env = df_env["Value"].values

In [ ]:
plt.plot(tem_env, label="env")
plt.plot(tem_sen, label='sen')
plt.legend()
plt.show()

In [ ]:
csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df_s = pd.read_csv(csv_semi_processed_path, encoding='utf-8')
df_s['date'] = pd.to_datetime(df_s['date'])
df_s['datetime'] = df_s['date'] + pd.to_timedelta(df_s['hour'], unit='h')

In [ ]:
from src.visualization.plotUnit import UnitPlotter

In [ ]:
df_s2 = df_s[(df_s['name'] == "پرند") & (df_s['code'] == 'G11') & (df_s["is_good_peak"] >= 3)]
df_merged = pd.merge(df_s2, df_sen, on="datetime", how="inner")
print(len(df_merged), len(df_s2))
my_cols = df_merged[['date', 'hour', 'temperature', 'Value', 'generation']].dropna()
#my_cols.dropna(inplace=True)
gens = my_cols['generation'].values
temp_senses = my_cols['Value'].values
plt.scatter(temp_senses, gens, s=0.5)
plt.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

power_plants = df_s[['name', 'code']].drop_duplicates()
for _, row in power_plants.iterrows():
    name, code = row['name'], row['code']

    # فیلتر کردن داده‌ها
    df_s2 = df_s[(df_s['name'] == name) & (df_s['code'] == code) & (df_s["is_good_peak"] >= 3)]
    df_merged = pd.merge(df_s2, df_sen, on="datetime", how="inner")
    print(len(df_merged), len(df_s2))

    # انتخاب ستون‌های مورد نیاز و حذف مقادیر NaN
    my_cols = df_merged[['date', 'hour', 'temperature', 'Value', 'generation']].dropna()
    gens = my_cols['generation'].values
    temp_senses = my_cols['Value'].values

    # ایجاد نمودار پراکندگی
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=temp_senses,
        y=gens,
        mode='markers',
        marker=dict(
            size=3,
            opacity=0.6,
            color='blue'
        ),
        name='داده‌های تولید'
    ))

    fig.update_layout(
        title=f'نمودار پراکندگی دمای حس شده vs تولید - {name+"-"+code}',
        xaxis_title='دمای توربین',
        yaxis_title='تولید (مگاوات)',
        template='plotly_white'
    )

    fig.show()

In [ ]:

import plotly.express as px
import plotly.graph_objects as go

power_plants = df_s[['name', 'code']].drop_duplicates()
for _, row in power_plants.iterrows():
    name, code = row['name'], row['code']

    # فیلتر کردن داده‌ها
    df_s2 = df_s[(df_s['name'] == name) & (df_s['code'] == code) & (df_s["is_good_peak"] >= 3)]
    df_merged = pd.merge(df_s2, df_sen, on="datetime", how="inner")
    print(len(df_merged), len(df_s2))

    # انتخاب ستون‌های مورد نیاز و حذف مقادیر NaN
    my_cols = df_merged[['date', 'hour', 'temperature', 'Value', 'generation']].dropna()
    gens = my_cols['generation'].values
    temp_senses = my_cols['Value'].values
    my_cols['year'] = pd.to_datetime(my_cols['date']).dt.year

    # ایجاد نمودار پراکندگی
    fig = go.Figure()

    fig = px.scatter(
    my_cols,
    x='Value',
    y='generation',
    color='year',
    title=f'نمودار پراکندگی دمای حس شده vs تولید - {name+"-"+code}',
    labels={'Value': 'دمای توربین', 'generation': 'تولید (مگاوات)', 'year': 'سال'},
    hover_data=['date', 'hour', 'temperature'],
    category_orders={'year': ['2021', '2022', '2023', '2024', '2025']}  # ترتیب سال‌ها
    )

    fig.update_traces(marker=dict(size=5, opacity=0.7, line=dict(width=0.5, color='DarkSlateGrey')))

    fig.update_layout(
        font_family='Tahoma',
        title_font_size=16
    )

    fig.write_html(f"{project_root}/src/visualization/unit_figs/generation_vs_temeperature_color/{name}-{code}.html")

1542 1707
1530 1665
1517 1707
1958 2139
3131 4597
